# ToolFinder Step-by-Step Walkthrough

This notebook demonstrates how semantic routing works in ToolFinder without requiring a live MCP server first.

You will:
1. Build a small synthetic tool catalog
2. Index tools with FAISS via `UniversalMCPRouter`
3. Route natural-language queries to top-k tools
4. Trigger and inspect no-route behavior

## Prerequisites

Run this once in your environment before opening the notebook:
- `pip install -e .[langgraph,dev]`

Note: the first run may download the sentence-transformers model.

In [ ]:
from __future__ import annotations

import json
from pprint import pprint

from toolfinder.dynamic_faiss_router import RouteNotFoundError, UniversalMCPRouter

In [ ]:
# Synthetic MCP-like tool schemas from two logical servers
filesystem_tools = [
    {
        "tool_name": "read_text_file",
        "description": "Read text content from a file path.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "tool_name": "write_file",
        "description": "Write text content to a file path.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["path", "content"]
        }
    },
    {
        "tool_name": "list_directory",
        "description": "List files and folders in a directory.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    }
]

memory_tools = [
    {
        "tool_name": "create_entities",
        "description": "Create entities in a memory graph.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "entities": {"type": "array"}
            },
            "required": ["entities"]
        }
    },
    {
        "tool_name": "read_graph",
        "description": "Read current entities and relations from memory graph.",
        "inputSchema": {
            "type": "object",
            "properties": {}
        }
    }
]

In [ ]:
router = UniversalMCPRouter(model_name="sentence-transformers/all-mpnet-base-v2")
router.ingest_server("filesystem", filesystem_tools)
router.ingest_server("memory", memory_tools)

print(f"Indexed tools: {router.faiss_index.ntotal}")

In [ ]:
def show_top_k(query: str, k: int = 2, min_score: float = 0.15) -> None:
    print(f"\nQUERY: {query}")
    matches = router.route_top_k(query, k=k, min_score=min_score)
    if not matches:
        print("No matches over threshold.")
        return
    for idx, match in enumerate(matches, start=1):
        print(f"{idx}. {match.server_name}/{match.tool_name}  score={match.score:.4f}")

show_top_k("read the sandbox input file")
show_top_k("store a note in memory graph")
show_top_k("list files in the folder")

In [ ]:
# Single-best route and strict no-route behavior
best = router.route("write success into output.txt")
print("Best route:", f"{best.server_name}/{best.tool_name}", f"score={best.score:.4f}")

try:
    router.route_top_k("totally unrelated quantum gardening tax policy", k=2, min_score=0.95)
    router.route("totally unrelated quantum gardening tax policy")
except RouteNotFoundError as exc:
    print("Caught RouteNotFoundError as expected:")
    print(str(exc))

## Next: run full script examples

From repository root, execute:
- `python examples/eval_toolfinder.py`
- `python examples/eval_toolfinder.py --update-readme` (optional README mutation)
- `python -u examples/langgraph_integration/baseline_agent.py`
- `python -u examples/langgraph_integration/benchmark_agent.py`
- `python -u examples/tri_server_demo.py`
- `python -u examples/verify_react_agent.py`